# 01 — Data Exploration & Week 1 Summary

Explores the synthetic vitals dataset (`src/data/synth_data.py`) and the engineered/split output (`src/data/feature_engineering.py`). Harespod is not yet downloaded (manual Figshare step) -- see `src/data/harespod_loader.py` for the ready adapter and CLAUDE.md Section 5 for the plan once it lands.

Run `python -m src.data.synth_data` then `python -m src.data.feature_engineering` from the project root before running this notebook, so `data/synthetic/` and `data/processed/` are populated.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `import src...` works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.config import PROCESSED_DIR, SYNTHETIC_DIR, SEVERITY_TIERS

raw = pd.read_csv(SYNTHETIC_DIR / "synthetic_vitals.csv")
train = pd.read_csv(PROCESSED_DIR / "train.csv")
val = pd.read_csv(PROCESSED_DIR / "val.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")
raw.shape, train.shape, val.shape, test.shape

## Class balance across the temporal, tier-stratified split

Every tier must appear in every split (this is what `tests/test_pipeline.py::test_temporal_split_no_subject_overlap` guards against regressing) -- an earlier version of the split used a plain chronological cut across all subjects and, by bad luck of subject ordering, produced a **test set with zero HACE-risk examples**. Stratifying the temporal cut per severity tier (still no row-level shuffling -- see `feature_engineering.temporal_split`'s docstring) fixed this.

In [ ]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(name, df["subject_id"].nunique(), "subjects")
    print(df["severity_label"].value_counts())
    print()

## Example trajectories per tier

Sanity check that onset is gradual (sigmoid ramp), not a step function -- see `synth_data.py`'s `_smooth_ramp`.

In [ ]:
fig, axes = plt.subplots(len(SEVERITY_TIERS), 1, figsize=(9, 12), sharex=True)
for ax, tier in zip(axes, SEVERITY_TIERS):
    subj = raw[raw["severity_label"] == tier]["subject_id"].iloc[0]
    traj = raw[raw["subject_id"] == subj].sort_values("timestamp")
    ax.plot(traj["timestamp"] / 60, traj["spo2"], label="SpO2")
    ax2 = ax.twinx()
    ax2.plot(traj["timestamp"] / 60, traj["altitude"], color="orange", alpha=0.5, label="altitude")
    ax.set_title(f"{tier} — example trajectory ({subj})")
    ax.set_ylabel("SpO2 (%)")
axes[-1].set_xlabel("minutes")
fig.tight_layout()

## Model comparison recap (Days 4-7)

See `src/models/artifacts/model_comparison.json` (written by `python -m src.models.predict_severity`) for the machine-readable record. As of the last run:

| Model | Precision (macro) | Recall (macro) | F1 (macro) | Mean Abs Tier Error | Under-triage rate |
|---|---|---|---|---|---|
| Rule-Based Baseline | 0.349 | 0.287 | 0.285 | 0.819 | 0.309 |
| XGBoost Ordinal | 0.387 | 0.405 | 0.374 | **0.594** | 0.221 |
| LSTM | 0.405 | 0.370 | 0.374 | 0.785 | 0.235 |

**XGBoost Ordinal selected** (lowest mean-abs-tier-error, clearly beats the rule-based sanity floor). Feature importances show `spo2_delta` and `time_at_altitude_min` dominate -- consistent with `docs/lls_mapping.md`'s reasoning that deviation-from-altitude-expected SpO2 (not raw SpO2) and cumulative exposure time are the clinically meaningful signals, not any single instantaneous reading.